In [4]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tqdm import tqdm
from PIL import Image
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Conv2D,
    Conv2DTranspose,
    Dense,
    Reshape,
    Embedding,
    BatchNormalization,
)

tf.keras.backend.clear_session()

# =============================================================================
# 1. Configuration Constants
# =============================================================================
CLASS_NAME = "NORM" 
NUM_IMAGES_TO_GENERATE = 2237
BATCH_SIZE = 32
LATENT_DIM = 128
N_CLASSES = 8 

SAVE_DIR = f"/kaggle/working/synthetic_data/{CLASS_NAME}/"
MODEL_WEIGHTS_PATH = "/kaggle/input/models/snowbubble0/gen-90/keras/default/1/generator_epoch_90.weights.h5"
ZIP_OUTPUT_NAME = f"/kaggle/working/synthetic_{CLASS_NAME}_images"

# Sorted alphabetically
CLASS_LABEL_MAP = {
    "ADI": 0, "DEB": 1, "LYM": 2, "MUC": 3, 
    "MUS": 4, "NORM": 5, "STR": 6, "TUM": 7
}
CLASS_LABEL_ID = CLASS_LABEL_MAP[CLASS_NAME]

os.makedirs(SAVE_DIR, exist_ok=True)

# =============================================================================
# 2. Generator Architecture
# =============================================================================
class ConditionalGenerator(Model):
    def __init__(self, n_classes, latent_dim, embed_dim=128, base_channels=512, img_channels=3, **kwargs):
        super().__init__(**kwargs)
        
        self.label_embedding = Embedding(n_classes, embed_dim, name="label_embedding")
        self.fc = Dense(4 * 4 * base_channels, name="fc")
        self.reshape = Reshape((4, 4, base_channels), name="reshape")

        self.g_tconv_8 = Conv2DTranspose(512, 4, strides=2, padding="same", use_bias=False, name="g_tconv_8")
        self.g_bn_8 = BatchNormalization(name="g_bn_8")
        
        self.g_tconv_16 = Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False, name="g_tconv_16")
        self.g_bn_16 = BatchNormalization(name="g_bn_16")
        
        self.g_tconv_32 = Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False, name="g_tconv_32")
        self.g_bn_32 = BatchNormalization(name="g_bn_32")
        
        self.g_tconv_64 = Conv2DTranspose(64, 4, strides=2, padding="same", use_bias=False, name="g_tconv_64")
        self.g_bn_64 = BatchNormalization(name="g_bn_64")
        
        self.g_tconv_128 = Conv2DTranspose(32, 4, strides=2, padding="same", use_bias=False, name="g_tconv_128")
        self.g_bn_128 = BatchNormalization(name="g_bn_128")

        self.g_out = Conv2D(img_channels, 7, padding="same", activation="tanh", name="g_out")

    def call(self, inputs, training=False):
        z, label = inputs
        le = self.label_embedding(label)
        le = tf.squeeze(le, axis=1)
        x = tf.concat([z, le], axis=-1)
        x = self.fc(x)
        x = self.reshape(x)

        x = tf.nn.relu(self.g_bn_8(self.g_tconv_8(x), training=training))
        x = tf.nn.relu(self.g_bn_16(self.g_tconv_16(x), training=training))
        x = tf.nn.relu(self.g_bn_32(self.g_tconv_32(x), training=training))
        x = tf.nn.relu(self.g_bn_64(self.g_tconv_64(x), training=training))
        x = tf.nn.relu(self.g_bn_128(self.g_tconv_128(x), training=training))
        return self.g_out(x)



In [5]:
# =============================================================================
# 3. Initialization & Weight Loading
# =============================================================================
print(f"Loading cGAN generator weights for {CLASS_NAME} (Label ID: {CLASS_LABEL_ID})...")
model = ConditionalGenerator(n_classes=N_CLASSES, latent_dim=LATENT_DIM)

# Dummy pass to initialize
dummy_z = tf.zeros((1, LATENT_DIM))
dummy_label = tf.zeros((1, 1), dtype=tf.int32)
_ = model([dummy_z, dummy_label], training=False)

# Load the weights
model.load_weights(MODEL_WEIGHTS_PATH)
print("Weights loaded successfully!\n")
# =============================================================================
# 4. Generation & Saving Loop
# =============================================================================
print(f"Generating {NUM_IMAGES_TO_GENERATE} synthetic {CLASS_NAME} images...")
num_batches = int(np.ceil(NUM_IMAGES_TO_GENERATE / BATCH_SIZE))
images_saved = 0

for i in tqdm(range(num_batches), desc="Generating Batches"):
    current_batch_size = min(BATCH_SIZE, NUM_IMAGES_TO_GENERATE - images_saved)
    
    # Generate random noise and class labels
    z = tf.random.normal((current_batch_size, LATENT_DIM))
    labels = tf.fill((current_batch_size, 1), CLASS_LABEL_ID)
    
    # Generate images (training=True bypasses bad Batch Normalization stats)
    fake_images = model([z, labels], training=True).numpy()
    
    for j in range(current_batch_size):
        # Safely convert from [-1, 1] scale to [0, 255] RGB scale
        img_normalized = (fake_images[j] + 1.0) / 2.0
        img_uint8 = np.clip(img_normalized * 255.0, 0, 255).astype(np.uint8)
        
        # Save using PIL
        pil_img = Image.fromarray(img_uint8)
        filename = f"synthetic_{CLASS_NAME}_{images_saved + 1:05d}.png"
        filepath = os.path.join(SAVE_DIR, filename)
        pil_img.save(filepath)
        
        images_saved += 1

print(f"\n{images_saved} images saved to: {SAVE_DIR}")


Loading cGAN generator weights for NORM (Label ID: 5)...
Weights loaded successfully!

Generating 2237 synthetic NORM images...


Generating Batches: 100%|██████████| 70/70 [00:09<00:00,  7.02it/s]


2237 images saved to: /kaggle/working/synthetic_data/NORM/


In [6]:
# =============================================================================
# 5. Compression Phase
# =============================================================================
print(f"Compressing images from {SAVE_DIR}...")
shutil.make_archive(ZIP_OUTPUT_NAME, 'zip', SAVE_DIR)
print(f"File is ready: {ZIP_OUTPUT_NAME}.zip")

Compressing images from /kaggle/working/synthetic_data/NORM/...
File is ready: /kaggle/working/synthetic_NORM_images.zip
